In [ ]:
import altair as alt
import pandas as pd

alt.data_transformers.disable_max_rows()

url = 'https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv'
df = pd.read_csv(url)

df['date'] = pd.to_datetime(df['date'], errors='coerce')

In [ ]:
plot1 = alt.Chart(df).mark_bar().encode(
    x=alt.X('season:N', title='Season', sort='-y'),
    y=alt.Y('count():Q', title='Number of Sightings'),
    color=alt.Color('season:N', legend=None)
).properties(
    width=400,
    height=300,
    title='Bigfoot Sightings by Season'
)

plot1.save('plot1.json')
plot1

In [ ]:
from vega_datasets import data
import pandas as pd
import altair as alt

seasons_list = [s for s in df['season'].unique() if pd.notna(s)]
seasons_list

season_dropdown = alt.binding_select(
    options=[None] + seasons_list,
    labels=['All Seasons'] + seasons_list,
    name='Filter by Season: '
)

season_select = alt.selection_point(fields=['season'], bind=season_dropdown)


states = alt.topo_feature(data.us_10m.url, feature='states')
background = alt.Chart(states).mark_geoshape(
    fill='#eeeeee',
    stroke='#bcbcbc'
).project(
    type='albersUsa'
).properties(
    width=600,
    height=400
)


domain_seasons = ['Spring', 'Summer', 'Fall', 'Winter', 'Unknown']
range_colors   = ['#2ca02c', '#d62728', '#ff7f0e', '#1f77b4', '#555555']

points = alt.Chart(df).mark_circle(size=20, opacity=0.7).encode(
    longitude='longitude:Q',
    latitude='latitude:Q',
    color=alt.condition(
        season_select,
        alt.Color('season:N',
                  scale=alt.Scale(domain=domain_seasons, range=range_colors),
                  title='Season'),
        alt.value('#dcdcdc')
    ),
    tooltip=['state:N', 'county:N', 'season:N', 'date:T']
).add_params(
    season_select
)


plot2 = (background + points).properties(
    title='Geographic Distribution of Bigfoot Sightings (Enhanced Visibility)'
)

plot2.save('plot2.json')
plot2